In [ ]:
from LanguageDatasets import LanguageDataset
from groq import Groq
from dotenv import load_dotenv
import os
import random
import time
import pandas as pd
import re
import math
from collections import Counter
from sacrebleu.metrics import BLEU, CHRF

### 1. Traducir un Q&A a los idiomas

### 2. Traducción como tarea de evaluación

La traducción asturiano ↔ español y aranés ↔ español permite medir comprensión y generación. Para el aranés puede ser útil incluir también comparaciones con francés, dado su parentesco occitano. 

In [ ]:
bleu = BLEU()
chrf = CHRF()

def translate(client, model, text):
    """
    Traduce un texto usando el modelo evaluado.
    """
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Traduce al español: {text}"}]
    )
    return response.choices[0].message.content.strip()

In [ ]:
def evaluate_translation(client, model, text):
    """
    1. Traduce el texto al español (referencia)
    2. Traduce el texto al español otra vez (hipótesis)
    3. Calcula BLEU y chrF entre ambas traducciones
    """
    # Traducción de referencia
    ref = translate(client, model, text)

    # Traducción hipótesis (puede ser otra instrucción o el mismo prompt)
    hyp = translate(client, model, text)

    # SacreBLEU espera listas
    bleu_score = bleu.corpus_score([hyp], [[ref]]).score
    chrf_score = chrf.corpus_score([hyp], [[ref]]).score

    return {
        "reference": ref,
        "hypothesis": hyp,
        "BLEU": bleu_score,
        "chrF": chrf_score
    }

In [ ]:
def round_trip_evaluation(client, model, text, lang_original):
    """
    1. Traduce del idioma original al español
    2. Traduce del español de vuelta al idioma original
    3. Compara original vs. vuelta con BLEU y chrF
    """
    # 1. Ida
    esp = translate(client, model, text, "español")

    # 2. Vuelta
    vuelta = translate(client, model, esp, lang_original)

    # 3. Métricas
    bleu_score = bleu.corpus_score([vuelta], [[text]]).score
    chrf_score = chrf.corpus_score([vuelta], [[text]]).score

    return {
        "original": text,
        "español": esp,
        "vuelta": vuelta,
        "BLEU": bleu_score,
        "chrF": chrf_score
    }


### 3. Corrección gramatical mediante introducción de errores

Para evaluar la capacidad de corrección, se parte de textos correctos en asturiano o aranés (por ejemplo, de Wikipedia). Un LLM grande introduce errores controlados de ortografía, morfología o sintaxis. El modelo evaluado debe corregirlos. La comparación con el texto original permite medir la calidad de la corrección.

- `"llama-3.3-70b-versatile"` Mucho más rápido
- `"openai/gpt-oss-120b"` Para producción?

In [9]:
def safe_chat_completion(client, model, messages, sleep_time=0.5, max_retries=5): 
    """ Llama a client.chat.completions.create con reintentos automáticos. Si falla (por ejemplo, error 429 o timeout), espera sleep_time y reintenta. """ 
    for attempt in range(max_retries): 
        try: 
            return client.chat.completions.create( model=model, messages=messages ) 
        except Exception as e: # Último intento → relanzar error 
            if attempt == max_retries - 1: 
                raise e # Espera antes del siguiente intento 
            wait = sleep_time * (attempt + 1) # backoff lineal 
            print(f"Error en intento {attempt+1}: {e}. Reintentando en {wait} segundos...") 
            time.sleep(wait)


In [ ]:
def generateDatasetOrtografico(dataset: LanguageDataset, api_key, model="openai/gpt-oss-120b", save = True, max_errors=4, min_errors=0, sleep_time=0.15, max_retries=5):
    client = Groq(api_key=api_key)
    res_list = []
    i = 0
    total = len(dataset)
    for original in dataset:
        callBegin = time.time()
        n_errors = random.randint(min_errors, max_errors)
        try:
            modified = safe_chat_completion(client, sleep_time=sleep_time, max_retries=max_retries,
                model=model,
                messages=[
                    {"role": "user", "content": f"Modifica esta frase, añadiendole {n_errors} errores gramaticales, ortográficos o léxicos. No añadas más contenido a la frase ni cambies el significado. Devuelve solo la frase modificada: '{original["text"]}'"}
                ]
            )
            res_list.append((original["text"], modified.choices[0].message.content, n_errors))
            callEnd = time.time()
        except Exception as e: # No perder todo lo conseguido si hay token limit o keybord interrupt
            print(e)
            return pd.DataFrame(res_list, columns=["original","modified", "n_errors"])
        # ---- PROGRESO ---- 
        i += 1
        pct = (i / total) * 100 
        step = int(callEnd - callBegin)
        remaining = step * (total - i)
        print(f"\rProgreso: {pct:5.1f}% ({i}/{total}) | step: {step} s, remaining time: {remaining// 60} min y {remaining - 60 * (remaining // 60)} s", end="") 
    print("\rDataset Generado") # salto de línea al terminar

    res_df = pd.DataFrame(res_list, columns=["original","modified", "n_errors"])
    if save:
        date = time.localtime(time.time())
        res_df.to_csv(f"{dataset.language}_{model.split("/")[-1]}_{time.strftime("%m-%d_%H-%M-%S", date)}")
    return res_df

In [15]:
load_dotenv("secrets.env")
ast = LanguageDataset("asturiano",True)
evalDataset = generateDatasetOrtografico(ast, os.getenv("GROQ_API_KEY"), save=False, model="llama-3.3-70b-versatile")

Descargando tatoeba para asturiano:
Completado con éxito
Progreso:  99.1% (444/448) | step: 2 s, remaining time: 0 min y 8 sssError en intento 1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99989, Requested 99. Please try again in 1m16.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}. Reintentando en 0.15 segundos...
Error en intento 2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99989, Requested 99. Please try again in 1m16.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limi

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kdjbyd5aeftbs2nrxfpm9e93` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 99987, Requested 99. Please try again in 1m14.304s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
evalDataset.shape

(448, 3)

In [ ]:
evalDataset.head()

,original,modified,1
0,Tas buenu pa dir a nengún sitiu.,"'Tas buenu pa dir a nengun sitiou, porque toi ...",4
1,El cielu del atapecer ye roxu.,"El cielu del atardecer es roxo, el barco navev...",3
2,Nun gastes más perres de les que ganes.,'Nun gastis más perros de les que ganas.',4
3,Lleva-y les llaves al to hermanu.,Lleva-y les llabe al tu hermano.,2
4,Esti xergón ye vieyu y máncame nel renaz.,Esti xergon ye viejyo y mancame en el renaz y ...,2


In [ ]:
evalDataset.n_errors.value_counts()

### 4. Medición de castellanización o interferencia lingüística

Para detectar si el modelo mezcla castellano con asturiano o aranés

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

In [ ]:
def merge_freeling_dictionary_fast(folder_path, output_file):
    """
    Combina todos los archivos .src del diccionario de FreeLing
    en un único archivo con una palabra por línea.
    Optimizado para velocidad.
    """
    dictionary_path = os.path.join(folder_path, "dictionary")
    lexicon = set()

    for filename in os.listdir(dictionary_path):

        full_path = os.path.join(dictionary_path, filename)

        with open(full_path, "r", encoding="utf8", errors="ignore") as f:
            for line in f:
                if not line or line.startswith("#"):
                    continue
                word = line.split(" ", 1)[0].lower() 
                lexicon.add(word)

    with open(output_file, "w", encoding="utf8") as out:
        out.write("\n".join(sorted(lexicon)))

    print(f"Lexicón generado: {output_file} ({len(lexicon)} palabras)")


In [ ]:
def loadLexicon(file_path):
    with open(file_path, "r", encoding="utf8") as f:
        return set(f.read().splitlines())

#### 4.1. Índice tipo‑token (TTR)

Se calcula como número de palabras únicas dividido entre el total de palabras. Un TTR bajo puede indicar uso excesivo de vocabulario castellano básico.

In [ ]:
def ttr(text):
    tokens = tokenize(text)
    if not tokens:
        return 0
    return len(set(tokens)) / len(tokens)


#### 4.2. Entropía léxica

Se calcula la distribución de frecuencias de los tokens y su entropía. Una entropía baja sugiere un vocabulario poco variado y potencial castellanización.

In [ ]:
def lexical_entropy(text):
    tokens = tokenize(text)
    if not tokens:
        return 0.0

    freqs = Counter(tokens)
    total = len(tokens)

    # precalcular 1/total para evitar divisiones repetidas
    inv_total = 1 / total

    entropy = 0.0
    for count in freqs.values():
        p = count * inv_total
        entropy -= p * math.log2(p)

    return entropy


#### 4.3. Frecuencia relativa de formas propias

Se construye un lexicón asturiano o aranés a partir de corpus públicos. Se compara la proporción de tokens generados por el modelo que pertenecen al lexicón propio frente a un lexicón castellano o francés (en el caso del aranés). Esto permite medir interferencia.

In [ ]:
def relative_language_frequency(text, lexicon_target, lexicon_spanish):
    tokens = tokenize(text)

    target_count = sum(1 for t in tokens if t in lexicon_target)
    spanish_count = sum(1 for t in tokens if t in lexicon_spanish)

    total = len(tokens)
    if total == 0:
        return 0, 0

    return target_count / total, spanish_count / total


#### 4.4. N‑gram overlap con corpus de referencia

Se toma un corpus real en asturiano o aranés (por ejemplo, Wikipedia). Se extraen sus n‑gramas y se comparan con los n‑gramas generados por el modelo. El modelo no recibe nada en esta fase; simplemente se analizan sus salidas. Un solapamiento bajo indica que el modelo no reproduce patrones característicos de la lengua.


In [ ]:
def ngrams(tokens, n):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def ngram_overlap(text, reference_text, n=3):
    tokens_gen = tokenize(text)
    tokens_ref = tokenize(reference_text)

    ngrams_gen = ngrams(tokens_gen, n)
    ngrams_ref = ngrams(tokens_ref, n)

    if not ngrams_gen:
        return 0

    overlap = ngrams_gen.intersection(ngrams_ref)
    return len(overlap) / len(ngrams_gen)
